# ** ĐỒ ÁN DRONE ADHOC**

## PHẦN 1: Cài đặt cơ bản

In [1]:
import time
import numpy as np
import pybullet as p
from gymnasium import spaces    
from gym_pybullet_drones.envs.BaseAviary import BaseAviary
from gym_pybullet_drones.utils.enums import DroneModel, Physics
from gym_pybullet_drones.control.DSLPIDControl import DSLPIDControl
from scipy.spatial import ConvexHull

c:\Users\Admin\anaconda3\envs\nt549-5-Tho\Lib\site-packages\gym_pybullet_drones\envs\BaseAviary.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


##### Enviroment mô phỏng

In [ ]:
class CustomAdHocEnv(BaseAviary):
    def __init__(self, 
                 num_drones=5, 
                 ad_hoc_radius=1.5, 
                 initial_xyzs=None, 
                 gui=True):
        
        # Thiết lập các tham số riêng cho bài toán
        self.AD_HOC_RADIUS = ad_hoc_radius
        self.TARGET_Z = 1.0  # Giữ các drone ở độ cao cố định để tối ưu 2D
        
        # Khởi tạo lớp cha (BaseAviary)
        super().__init__(drone_model=DroneModel.CF2X,
                         num_drones=num_drones,
                         initial_xyzs=initial_xyzs,
                         physics=Physics.PYB,
                         pyb_freq=240,
                         ctrl_freq=48,
                         gui=gui,
                         )
    def _actionSpace(self):
        """Định nghĩa không gian hành động: 4 motor RPM cho mỗi drone"""
        # Mỗi drone nhận 4 giá trị RPM (thường từ 0 đến 65535, tùy model)
        # Ở đây ta dùng giới hạn của Crazyflie
        low = np.zeros(4)
        high = np.ones(4) * 60000 
        # Tạo mảng action space cho NUM_DRONES con
        return spaces.Box(low=np.tile(low, (self.NUM_DRONES, 1)), 
                          high=np.tile(high, (self.NUM_DRONES, 1)), 
                          dtype=np.float32)

    def _observationSpace(self):
        """Định nghĩa không gian quan sát: 20 chỉ số trạng thái cho mỗi drone"""
        # 1 drone state gồm: pos(3), quarternion(4), rpy(3), vel(3), ang_vel(3), last_action(4) = 20
        low = np.ones(20) * -np.inf
        high = np.ones(20) * np.inf
        return spaces.Box(low=np.tile(low, (self.NUM_DRONES, 1)), 
                          high=np.tile(high, (self.NUM_DRONES, 1)), 
                          dtype=np.float32)
    def _computeObs(self):
        """Tự định nghĩa State: Drone nhìn thấy gì?"""
        # Trả về vị trí và vận tốc của tất cả drone
        return np.array([self._getDroneStateVector(i) for i in range(self.NUM_DRONES)])

    def _computeTerminated(self):
        """Khi nào thì dừng mô phỏng?"""
        return False # Chạy liên tục cho đến khi hết bước

    def _computeTruncated(self):
        return False

    def _computeInfo(self):
        return {"area": self.get_coverage_area(), "connected": self.is_connected()}
        
    def get_adjacency_matrix(self):
        """Tính toán ma trận kề của mạng Ad-hoc"""
        pos = np.array([self._getDroneStateVector(i)[0:3] for i in range(self.NUM_DRONES)])
        adj = np.zeros((self.NUM_DRONES, self.NUM_DRONES))
        for i in range(self.NUM_DRONES):
            for j in range(i + 1, self.NUM_DRONES):
                dist = np.linalg.norm(pos[i] - pos[j])
                if dist <= self.AD_HOC_RADIUS:
                    adj[i][j] = adj[j][i] = 1
        return adj

    def is_connected(self):
        """Kiểm tra xem toàn bộ mạng có liên thông hay không (dùng BFS)"""
        adj = self.get_adjacency_matrix()
        visited = [False] * self.NUM_DRONES
        queue = [0]
        visited[0] = True
        while queue:
            u = queue.pop(0)
            for v, connected in enumerate(adj[u]):
                if connected and not visited[v]:
                    visited[v] = True
                    queue.append(v)
        return all(visited)

    def get_coverage_area(self):
        """Tính diện tích bao phủ bằng Convex Hull (Bao lồi)"""
        if self.NUM_DRONES < 3: return 0
        pos = np.array([self._getDroneStateVector(i)[0:2] for i in range(self.NUM_DRONES)]) # Xét mặt phẳng 2D
        try:
            hull = ConvexHull(pos)
            return hull.area # Trong 2D, .area trả về chu vi, .volume trả về diện tích
        except:
            return 0

##### hàm reward

In [3]:
def _computeReward(self):
        """Tự định nghĩa Reward: Drone được thưởng vì điều gì?"""
        # 1. Thưởng cho diện tích bao phủ
        area = self.get_coverage_area()
        
        # 2. Phạt nếu mất kết nối
        connectivity_bonus = 10 if self.is_connected() else -50
        
        # 3. Phạt nếu va chạm (khoảng cách giữa các drone < 0.3m)
        collision_penalty = 0
        adj = self.get_adjacency_matrix(radius=0.3)
        if np.sum(adj) > 0:
            collision_penalty = -100
            
        return area + connectivity_bonus + collision_penalty

In [4]:
env = CustomAdHocEnv(num_drones=6, ad_hoc_radius=2.0, gui=True)
controllers = [DSLPIDControl(drone_model=DroneModel.CF2X) for _ in range(6)]

obs, info = env.reset()

for i in range(10000):
    actions = []
    # Tại đây bạn có thể gọi model RL: action = model.predict(obs)
    # Hoặc dùng bộ điều khiển PID như cũ để test môi trường
    for j in range(6):
        # Ví dụ: cho drone bay lơ lửng tại chỗ để test môi trường
        action, _, _ = controllers[j].computeControlFromState(
            control_timestep=env.CTRL_TIMESTEP,
            state=obs[j],
            target_pos=np.array([j*0.5, 0, 1.0]) 
        )
        actions.append(action)
    
    obs, reward, terminated, truncated, info = env.step(np.array(actions))
    
    if i % 50 == 0:
        print(f"Reward: {reward:.2f} | Area: {info['area']:.2f}")
    
    env.render()
    time.sleep(env.CTRL_TIMESTEP)

env.close()

[INFO] BaseAviary.__init__() loaded parameters from the drone's .urdf:
[INFO] m 0.027000, L 0.039700,
[INFO] ixx 0.000014, iyy 0.000014, izz 0.000022,
[INFO] kf 3.160000e-10, km 7.940000e-12,
[INFO] t2w 2.250000, max_speed_kmh 30.000000,
[INFO] gnd_eff_coeff 11.368590, prop_radius 0.023135,
[INFO] drag_xy_coeff 0.000001, drag_z_coeff 0.000001,
[INFO] dw_coeff_1 2267.180000, dw_coeff_2 0.160000, dw_coeff_3 -0.110000
viewMatrix (0.642787516117096, -0.4393851161003113, 0.6275069713592529, 0.0, 0.766044557094574, 0.36868777871131897, -0.5265407562255859, 0.0, -0.0, 0.8191521167755127, 0.5735764503479004, 0.0, 2.384185791015625e-07, 2.384185791015625e-07, -5.000000476837158, 1.0)
projectionMatrix (0.7499999403953552, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, -1.0000200271606445, -1.0, 0.0, 0.0, -0.02000020071864128, 0.0)


c:\Users\Admin\anaconda3\envs\nt549-5-Tho\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\Admin\anaconda3\envs\nt549-5-Tho\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


NotImplementedError: 